#### Load and Preprocess the Data
- **Load Dataset**: Read the dataset from a CSV file.
- **Separate Features and Target**: Identify the target variable (e.g., 'PerformanceScore') and separate it from the features.
- **Categorize Columns**: Distinguish between categorical and numerical columns in the dataset.
- **Preprocess Data**:
  - **StandardScaler**: Normalize numerical features to have mean 0 and variance 1.
  - **OneHotEncoder**: Convert categorical variables into a form that could be provided to ML algorithms (one-hot encoding).
- **Train-Test Split**: Split the dataset into training and test sets, typically using an 80-20 split.


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2

# Load dataset
file_path = 'Modified_IT_Project_Team_Member_Recommendation_Data.csv'
df = pd.read_csv(file_path)

# Assuming 'PerformanceScore' is the target variable
target = 'PerformanceScore'

# Separate features and target
X = df.drop(target, axis=1)
y = df[target].values

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object', 'category']).columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns

# Preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# Splitting data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply preprocessing
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)


#### Build the Model
- **Sequential Model**: Use Keras Sequential API to build the neural network layer by layer.
- **Dense Layers**: Add fully connected (Dense) layers with ReLU activation. Start with a higher number of neurons and gradually decrease.
- **Regularization**:
  - **L2 Regularization**: Add L2 regularization to each Dense layer to penalize large weights and prevent overfitting.
  - **Dropout Layers**: Include Dropout layers to randomly set a fraction of input units to 0 at each update during training, which helps prevent overfitting.
- **Output Layer**: The final Dense layer with a single neuron for regression output (no activation function or linear activation).


In [6]:
# Model architecture
model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],), kernel_regularizer=l2(0.001)))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.001)))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu', kernel_regularizer=l2(0.001)))
model.add(Dense(1))  # Output layer for regression


# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
history = model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2)




Epoch 1/100
20/20 [==============================] - 0s 4ms/step - loss: 28.7691 - val_loss: 21.8791
Epoch 2/100
20/20 [==============================] - 0s 1ms/step - loss: 14.1482 - val_loss: 8.9981
Epoch 3/100
20/20 [==============================] - 0s 1ms/step - loss: 11.8425 - val_loss: 9.2774
Epoch 4/100
20/20 [==============================] - 0s 1ms/step - loss: 10.7462 - val_loss: 9.2137
Epoch 5/100
20/20 [==============================] - 0s 1ms/step - loss: 9.9056 - val_loss: 9.1449
Epoch 6/100
20/20 [==============================] - 0s 1ms/step - loss: 11.1463 - val_loss: 9.2081
Epoch 7/100
20/20 [==============================] - 0s 1ms/step - loss: 10.4375 - val_loss: 8.9561
Epoch 8/100
20/20 [==============================] - 0s 1ms/step - loss: 9.8311 - val_loss: 9.1516
Epoch 9/100
20/20 [==============================] - 0s 1ms/step - loss: 10.2790 - val_loss: 9.2598
Epoch 10/100
20/20 [==============================] - 0s 1ms/step - loss: 9.8927 - val_loss: 8.8743
E

#### Compile and Train the Model
- **Compile the Model**:
  - **Optimizer**: Use 'adam' optimizer, which is an extension to stochastic gradient descent.
  - **Loss Function**: Use 'mean_squared_error' as it is a regression task.
- **Train the Model**:
  - **Epochs**: Set the number of epochs. This is the number of times the learning algorithm will work through the entire training dataset.
  - **Batch Size**: Set the batch size to define the number of samples that will be propagated through the network.
  - **Validation Split**: Use a part of the training data as a validation set to monitor the model's performance and prevent overfitting.


In [7]:
# Evaluate the model
test_loss = model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")


7/7 [==============================] - 0s 1ms/step - loss: 8.3581
Test Loss: 8.358138084411621


#### Evaluate the Model
- **Test Loss Evaluation**: After training, evaluate the model on the test set to check its performance on unseen data.
- **Output**: Display the test loss, which provides an estimate of the model’s prediction error.


In [9]:
# define the pipeline and save it
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Define the pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# Save the pipeline
import joblib
joblib.dump(pipeline, 'model.joblib')

# Load the pipeline
pipeline = joblib.load('model.joblib')

pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['Task_Duration', 'Member_Skill_Level', 'Member_Experience',
       'Member_Workload'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['Task_Difficulty', 'Task_Type'], dtype='object'))])),
                ('model',
                 <keras.src.engine.sequential.Sequential object at 0x15b94a620>)])